# CMS ROOT file access with XRootD and uproot

This notebook copies a CMS ROOT file with `xrdcp` using the mounted X.509 proxy, then opens the local copy with `uproot`. The Jupyter pod mounts the shared training PVC at `/training`; this notebook stores its copy under `/training/cms-data/notebook`.

In [ ]:
import os
from pathlib import Path

print("X509_USER_PROXY =", os.environ.get("X509_USER_PROXY"))
print("proxy exists     =", Path(os.environ["X509_USER_PROXY"]).exists())

In [ ]:
import os
import subprocess

remote = "root://eoscms.cern.ch//eos/cms/store/group/cmst3/group/l1tr/maglowac/AD_HLT_PF/QCD_Bin-Pt-15to7000_TuneCP5_13p6TeV_pythia8/re-emul_Run3Winter25MiniAOD-FEVTOUTPUT_142X_v7-v1/251124_134438/0000/nanoout_1.root"
local = Path(os.environ.get("LOCAL_ROOT_FILE", "/training/cms-data/notebook/nanoout_1.root"))
local.parent.mkdir(parents=True, exist_ok=True)

subprocess.run(["xrdcp", "-f", remote, str(local)], check=True)
print(local, local.stat().st_size, "bytes")

In [ ]:
import uproot

root_file = uproot.open(local)
root_file.keys()

In [ ]:
classnames = root_file.classnames()
tree_names = [name.split(";")[0] for name, class_name in classnames.items() if "TTree" in class_name]
tree_name = "Events" if "Events" in tree_names else tree_names[0]
events = root_file[tree_name]

print("tree:", tree_name)
print("entries:", events.num_entries)
events.keys()[:30]

In [ ]:
preferred = ["nMuon", "Muon_pt", "Muon_eta", "nJet", "Jet_pt", "Jet_eta", "MET_pt"]
available = events.keys()
branches = [branch for branch in preferred if branch in available]
if not branches:
    branches = available[:5]

arrays = events.arrays(branches, entry_stop=10)
arrays